In [1]:
# 01_run_baselines_classic.ipynb
# Multi-seed run of the non-generative baselines: None, SMOTE, ADASYN, G-SMOTE.
# Protocol: 5 repeated stratified 70/30 holdouts (SEEDS). Output: ../results/tables/raw_classic.csv
# ENV: scikit-learn, imbalanced-learn, imbalanced-learn-extra (imblearn_extra), lightgbm.
# ---- Shared configuration and protocol (identical across run notebooks) ----
# This block is self-contained so each notebook runs independently in its own
# conda environment. All 16 datasets must sit in ../data as <name>.csv with a
# 'target' column (1 = minority/positive).
import os, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score,
                             average_precision_score, balanced_accuracy_score, confusion_matrix)
warnings.filterwarnings("ignore")

DATA_DIR   = "../data"
TABLES_DIR = "../results/tables"
os.makedirs(TABLES_DIR, exist_ok=True)

# Evaluation protocol for every method: 5 repeated stratified 70/30 holdouts.
SEEDS = [42, 43, 44, 45, 46]
# Optional: set to a short list (e.g. ["ecoli", "pima_diabetes"]) to test quickly.
DATASETS_SUBSET = None

DATASETS = ["credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
            "yeast_me2", "mammography", "abalone_19", "wine_quality", "ecoli",
            "pageblocks", "protein_homo", "satellite", "churn", "secom",
            "thyroid_sick", "unsw_nb15"]
CLASSIFIERS = ("RF", "LGBM", "MLP")
RAW_COLUMNS = ["dataset", "method", "classifier", "seed", "AUC", "PR_AUC", "F1",
               "Precision", "Recall", "G_mean", "Balanced_Acc",
               "generation_time", "train_time", "method_group"]

def load_dataset(name):
    """Read ../data/<name>.csv -> (X float64 ndarray, y int ndarray)."""
    df = pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv"))
    bcols = df.select_dtypes(include="bool").columns
    if len(bcols):
        df[bcols] = df[bcols].astype("float64")
    X = df.drop(columns=["target"]).values.astype(float)
    y = df["target"].values.astype(int)
    return X, y

def get_classifier(name, variant, random_state):
    """variant: 'generative' (notebooks 03/04x config) or 'classic' (notebook 02)."""
    if name == "RF":
        return RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=3,
                                      class_weight="balanced", random_state=random_state, n_jobs=-1)
    if name == "LGBM":
        from lightgbm import LGBMClassifier
        if variant == "classic":
            return LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=31,
                                  random_state=random_state, verbosity=-1, n_jobs=-1)
        return LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=31,
                              class_weight="balanced", random_state=random_state, n_jobs=-1, verbose=-1)
    if name == "MLP":
        if variant == "classic":
            return MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu",
                                 alpha=0.001, random_state=random_state, max_iter=500)
        return MLPClassifier(hidden_layer_sizes=(128, 64), alpha=0.001,
                             max_iter=300, random_state=random_state)
    raise ValueError(name)

def g_mean_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return float(np.sqrt(sens * spec))

def evaluate(model, X_test, y_test):
    yp = model.predict_proba(X_test)[:, 1]
    yd = model.predict(X_test)
    return {"AUC": round(roc_auc_score(y_test, yp), 4),
            "PR_AUC": round(average_precision_score(y_test, yp), 4),
            "F1": round(f1_score(y_test, yd, average="macro", zero_division=0), 4),
            "Precision": round(precision_score(y_test, yd, average="macro", zero_division=0), 4),
            "Recall": round(recall_score(y_test, yd, average="macro", zero_division=0), 4),
            "G_mean": round(g_mean_score(y_test, yd), 4),
            "Balanced_Acc": round(balanced_accuracy_score(y_test, yd), 4)}

def holdout_split_scale(X, y, seed):
    """Single stratified 70/30 holdout; StandardScaler fit on train only."""
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, stratify=y, random_state=seed)
    sc = StandardScaler()
    return sc.fit_transform(Xtr), sc.transform(Xte), ytr, yte

def append_rows(out_csv, rows):
    pd.DataFrame(rows, columns=RAW_COLUMNS).to_csv(
        out_csv, mode="a", header=not os.path.exists(out_csv), index=False)

def seed_everything(seed):
    np.random.seed(seed)
    try:
        import torch; torch.manual_seed(seed)
    except Exception:
        pass

def run_oversampler(method, method_group, oversample_fn, out_csv, variant="generative",
                    skip_datasets=()):
    """Generic driver: for each seed x dataset, oversample then fit/evaluate 3 classifiers.
    oversample_fn(X_tr_scaled, y_tr, seed) -> (X_res, y_res, generation_time_or_None)."""
    datasets = DATASETS_SUBSET or DATASETS
    for seed in SEEDS:
        for ds in datasets:
            if ds in skip_datasets:
                continue
            try:
                X, y = load_dataset(ds)
            except FileNotFoundError:
                print(f"[missing csv] {ds}"); continue
            Xtr, Xte, ytr, yte = holdout_split_scale(X, y, seed)
            seed_everything(seed)
            t0 = time.time()
            Xr, yr, gt = oversample_fn(Xtr, ytr, seed)
            gen_time = round(time.time() - t0, 3) if gt is None else round(gt, 3)
            rows = []
            for cn in CLASSIFIERS:
                clf = get_classifier(cn, variant, random_state=seed)
                t1 = time.time(); clf.fit(Xr, yr); tt = round(time.time() - t1, 3)
                rows.append({"dataset": ds, "method": method, "classifier": cn, "seed": seed,
                             **evaluate(clf, Xte, yte), "generation_time": gen_time,
                             "train_time": tt, "method_group": method_group})
            append_rows(out_csv, rows)
            print(f"[ok] {method:12s} seed={seed} {ds:16s} "
                  f"AUC(RF/LGBM/MLP)={rows[0]['AUC']}/{rows[1]['AUC']}/{rows[2]['AUC']}")

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn_extra.gsmote import GeometricSMOTE

OUT = os.path.join(TABLES_DIR, "raw_classic.csv")

def make_sampler_fn(factory):
    def _fn(Xtr, ytr, seed):
        s = factory(seed)
        if s is None:
            return Xtr, ytr, None
        try:
            Xr, yr = s.fit_resample(Xtr, ytr); return Xr, yr, None
        except Exception as e:
            print("   [warn] resample failed:", e); return Xtr, ytr, None
    return _fn

for method, fac in {
    "None":    (lambda s: None),
    "SMOTE":   (lambda s: SMOTE(random_state=s)),
    "ADASYN":  (lambda s: ADASYN(random_state=s)),
    "G-SMOTE": (lambda s: GeometricSMOTE(random_state=s)),
}.items():
    print("=====", method, "=====")
    run_oversampler(method, "non_generative", make_sampler_fn(fac), OUT, variant="classic")
print("DONE -> ", OUT)


===== None =====
[ok] None         seed=42 credit_default   AUC(RF/LGBM/MLP)=0.7757/0.7786/0.6976
[ok] None         seed=42 fraud_detection  AUC(RF/LGBM/MLP)=0.9666/0.7689/0.952
[ok] None         seed=42 pima_diabetes    AUC(RF/LGBM/MLP)=0.8394/0.8158/0.7831
[ok] None         seed=42 ibm_attrition    AUC(RF/LGBM/MLP)=0.7796/0.7557/0.7471
[ok] None         seed=42 yeast_me2        AUC(RF/LGBM/MLP)=0.9111/0.8855/0.808
[ok] None         seed=42 mammography      AUC(RF/LGBM/MLP)=0.9707/0.9782/0.9747
[ok] None         seed=42 abalone_19       AUC(RF/LGBM/MLP)=0.8475/0.8371/0.8239
[ok] None         seed=42 wine_quality     AUC(RF/LGBM/MLP)=0.8821/0.8785/0.8118
[ok] None         seed=42 ecoli            AUC(RF/LGBM/MLP)=0.9576/0.9505/0.9162
[ok] None         seed=42 pageblocks       AUC(RF/LGBM/MLP)=0.9989/0.9989/0.9978
[ok] None         seed=42 protein_homo     AUC(RF/LGBM/MLP)=0.9952/0.9969/0.9947
[ok] None         seed=42 satellite        AUC(RF/LGBM/MLP)=0.9533/0.9576/0.9573
[ok] None    